In [ ]:
import os
import json
import re
import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai
from google.generativeai import types as genai_types
from unidecode import unidecode
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm # Para Jupyter
# from tqdm import tqdm # Para scripts Python normais (descomente e comente a de cima)
import time
from copy import deepcopy
import logging
import hashlib
import sys

# --- CONFIGURAÇÃO DE LOGGING ---
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - [%(module)s.%(funcName)s:%(lineno)d] - %(message)s',
                    handlers=[
                        logging.FileHandler("gemini_api_calls.log", mode='a'),
                        logging.StreamHandler()
                    ])
logger = logging.getLogger(__name__)

# --- CARREGAMENTO DINÂMICO DE PROMPTS ---
PROMPTS_ORACULO = {}
try:
    from activetextclassification.oraculo.prompts import prompts as prompts_carregados
    PROMPTS_ORACULO = prompts_carregados
    if not isinstance(PROMPTS_ORACULO, dict) or not PROMPTS_ORACULO:
        logger.error("A variável 'prompts' importada não é um dicionário válido ou está vazia.")
        PROMPTS_ORACULO = {}
    else:
        logger.info(f"Prompts carregados. Chaves: {list(PROMPTS_ORACULO.keys())}")
except ImportError as e:
    logger.error(f"Não foi possível importar 'prompts': {e}. Verifique PYTHONPATH/estrutura.")
except Exception as e:
    logger.exception(f"Erro inesperado ao carregar prompts.")

if not PROMPTS_ORACULO:
    logger.warning("PROMPTS_ORACULO vazio. Experimentos que dependem dele podem falhar.")

# Carregar variáveis de ambiente
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

# --- CAMINHOS DE ARQUIVOS ---
# CAMINHOS DE SAÍDA ATUALIZADOS PARA A PASTA data_oraculo
PATH_ARQUIVO_CONTROLE_EXCEL = r"D:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\data_oraculo\resultados_processamento_gemini.xlsx"
PATH_ARQUIVO_ERROS_MODELO_EXCEL = r"D:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\data_oraculo\erros_modelo_gemini.xlsx"
PATH_ARQUIVO_SUMARIO_METRICAS_EXCEL = r"D:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\data_oraculo\sumario_metricas_gemini.xlsx"

# Caminhos de entrada (permanecem os mesmos, a menos que você os mova também)
PATH_ARQUIVO_EXPERIMENTOS_JSON = r"D:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\data_oraculo\experimentos_oraculo.json"
PATH_BASE_CSV_DADOS_COLDSTART = r"D:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\data_splits_cache\dri_coldstart_selection_details_log.csv"
PATH_ARQUIVO_ALL_LABELS_JSON = r"D:\Nuvem\ghdaru\OneDrive\030_DOUTORADO\120_TESE\130_TESEGIT\activetextclassification\data_splits_cache\all_possible_labels_original.json"

# --- CONSTANTES GERAIS ---
STRING_ERRO_API_NA_EXPANDIDA = "ERRO API"
COLUNAS_RESULTADO_LLM_BASE = ["descrição", "descrição_expandida", "descricao_machine_learning", "categoria", "racional"]
COLUNAS_IDENTIFICADORES_EXPERIMENTO = ['prompt_version_key', 'target_run_id']
COLUNAS_METADADOS_API = ['modelo', 'temperatura_usada', 'prompt_tokens', 'completion_tokens', 'total_tokens', 'hash_processamento'] + COLUNAS_IDENTIFICADORES_EXPERIMENTO
COLUNAS_CONTROLE_COMPLETO = COLUNAS_RESULTADO_LLM_BASE + COLUNAS_METADADOS_API + ['ground_truth_categoria', 'predicao_categoria']

# --- CONFIGURAÇÃO API ---
API_CONFIGURADA_COM_SUCESSO = False
if not GOOGLE_API_KEY:
    logger.error("GEMINI_API_KEY não encontrada.")
else:
    try:
        genai.configure(api_key=GOOGLE_API_KEY)
        logger.info("API Key do Google Gemini configurada.")
        API_CONFIGURADA_COM_SUCESSO = True
    except Exception as e:
        logger.exception("Erro ao configurar API Key.")

# --- FUNÇÕES UTILITÁRIAS ---
def gerar_hash_processamento(descricao: str, modelo: str, temperatura: float, prompt_ver_key: str, target_run: str) -> str:
    temp_str = "{:.2f}".format(temperatura)
    hash_input = f"{descricao.strip()}|{modelo.strip()}|{temp_str}|{prompt_ver_key.strip()}|{target_run.strip()}"
    return hashlib.sha256(hash_input.encode('utf-8')).hexdigest()

def carregar_experimentos(caminho_json: str) -> list:
    try:
        with open(caminho_json, 'r', encoding='utf-8') as f:
            experimentos = json.load(f)
        logger.info(f"Carregados {len(experimentos)} experimentos de '{caminho_json}'.")
        valido = True
        for i, exp in enumerate(experimentos):
            if not all(k in exp for k in ["model_name", "temperature", "prompt_version_key", "target_run_id"]):
                logger.error(f"Experimento {i} em '{caminho_json}' malformado.")
                valido = False; break
            try: exp["temperature"] = float(exp["temperature"])
            except ValueError: logger.error(f"Temperatura inválida no exp {i}."); valido = False; break
        return experimentos if valido else []
    except FileNotFoundError: logger.error(f"Arquivo de experimentos '{caminho_json}' não encontrado."); return []
    except json.JSONDecodeError: logger.error(f"Erro ao decodificar JSON de experimentos '{caminho_json}'."); return []
    except Exception as e: logger.exception(f"Erro inesperado ao carregar experimentos de '{caminho_json}'."); return []

def carregar_lista_categorias_mestra(caminho_json_labels: str) -> list:
    try:
        with open(caminho_json_labels, 'r', encoding='utf-8') as f:
            lista_labels = json.load(f)
        if isinstance(lista_labels, list) and all(isinstance(label, str) for label in lista_labels):
            lista_labels_final = sorted(list(set(lista_labels + ["_RARE_"])))
            logger.info(f"Lista de categorias mestra carregada de '{caminho_json_labels}' com {len(lista_labels_final)} categorias.")
            return lista_labels_final
        else:
            logger.error(f"Conteúdo de '{caminho_json_labels}' não é uma lista de strings.")
            return ["_RARE_"] # Fallback
    except FileNotFoundError:
        logger.error(f"Arquivo de labels mestra '{caminho_json_labels}' não encontrado. Usando fallback ['_RARE_'].")
        return ["_RARE_"]
    except json.JSONDecodeError:
        logger.error(f"Erro ao decodificar JSON do arquivo de labels mestra '{caminho_json_labels}'. Usando fallback ['_RARE_'].")
        return ["_RARE_"]
    except Exception as e:
        logger.exception(f"Erro inesperado ao carregar lista de categorias mestra de '{caminho_json_labels}'.")
        return ["_RARE_"]

lista_categorias_py_mestra = carregar_lista_categorias_mestra(PATH_ARQUIVO_ALL_LABELS_JSON)
lista_categorias_str_mestra = json.dumps(lista_categorias_py_mestra, ensure_ascii=False)
if len(lista_categorias_py_mestra) <= 1 and "_RARE_" in lista_categorias_py_mestra:
    logger.warning("Lista de categorias mestra contém apenas '_RARE_' ou está vazia. Verifique o arquivo de labels.")


def carregar_resultados_anteriores_completos(caminho_arquivo: str) -> tuple[pd.DataFrame, dict, set]:
    df_controle_total = pd.DataFrame(columns=COLUNAS_CONTROLE_COMPLETO)
    map_hash_para_resultado_existente = {}
    set_hashes_com_erro_anterior = set()
    if os.path.exists(caminho_arquivo):
        try:
            df_temp = pd.read_excel(caminho_arquivo, dtype={'descrição': str})
            for col in COLUNAS_CONTROLE_COMPLETO:
                if col not in df_temp.columns:
                    default_val = 0 if col in ['prompt_tokens', 'completion_tokens', 'total_tokens'] else (
                                  "v_default_load" if col == "prompt_version_key" else (
                                  "id_default_load" if col == "target_run_id" else (
                                  0.0 if col == "temperatura_usada" else None)))
                    df_temp[col] = default_val
            df_controle_total = df_temp[COLUNAS_CONTROLE_COMPLETO]
            logger.info(f"Arquivo de controle '{caminho_arquivo}' carregado com {len(df_controle_total)} registros.")

            for index, row in df_controle_total.iterrows():
                current_hash_from_file = str(row.get('hash_processamento', '')).strip()
                desc_val = str(row.get('descrição','')).strip()
                modelo_val = str(row.get('modelo','')).strip()
                temp_val_raw = row.get('temperatura_usada')
                prompt_key_val = str(row.get('prompt_version_key','v_default_load')).strip()
                target_run_val = str(row.get('target_run_id','id_default_load')).strip()
                hash_calculado_da_linha = None
                if desc_val and modelo_val and pd.notna(temp_val_raw) and prompt_key_val and target_run_val:
                    try:
                        temp_val = float(temp_val_raw)
                        hash_calculado_da_linha = gerar_hash_processamento(desc_val, modelo_val, temp_val, prompt_key_val, target_run_val)
                    except ValueError: continue
                else: continue
                final_key_for_map = hash_calculado_da_linha
                if not final_key_for_map: continue
                if current_hash_from_file and current_hash_from_file != final_key_for_map:
                    logger.warning(f"Hash no arquivo ({current_hash_from_file[:7]}) difere do calculado ({final_key_for_map[:7]}) para desc '{desc_val[:30]}...'. Usando o calculado.")

                registro_mapa = row.to_dict()
                for col_token in ['prompt_tokens', 'completion_tokens', 'total_tokens']:
                    try: registro_mapa[col_token] = int(float(registro_mapa.get(col_token, 0))) if pd.notna(registro_mapa.get(col_token)) else 0
                    except: registro_mapa[col_token] = 0
                try: registro_mapa['temperatura_usada'] = float(registro_mapa.get('temperatura_usada', 0.0)) if pd.notna(registro_mapa.get('temperatura_usada')) else 0.0
                except: registro_mapa['temperatura_usada'] = 0.0
                if pd.isna(registro_mapa.get("categoria")) or not str(registro_mapa.get("categoria")).strip() : registro_mapa["categoria"] = "_RARE_"
                registro_mapa['hash_processamento'] = final_key_for_map
                map_hash_para_resultado_existente[final_key_for_map] = registro_mapa
                if STRING_ERRO_API_NA_EXPANDIDA in str(row.get('descrição_expandida', '')):
                    set_hashes_com_erro_anterior.add(final_key_for_map)
        except Exception as e:
            logger.exception(f"Erro CRÍTICO ao ler/processar controle completo '{caminho_arquivo}'.")
            return pd.DataFrame(columns=COLUNAS_CONTROLE_COMPLETO), {}, set()
    else:
        logger.info(f"Arquivo de controle '{caminho_arquivo}' não encontrado.")
    return df_controle_total, map_hash_para_resultado_existente, set_hashes_com_erro_anterior

def recalcular_e_atualizar_hashes_controle(caminho_arquivo_excel: str,
                                          default_prompt_key_retro: str,
                                          default_target_run_id_retro: str,
                                          default_temperatura_retro: float,
                                          default_modelo_retro: str):
    logger.info(f"Iniciando RECALCULATE e ATUALIZAÇÃO de hashes/colunas para '{caminho_arquivo_excel}'.")
    if not os.path.exists(caminho_arquivo_excel):
        logger.error(f"Arquivo para RECALCULATE de hash não encontrado: {caminho_arquivo_excel}"); return False
    try:
        df = pd.read_excel(caminho_arquivo_excel, dtype={'descrição': str})
        colunas_para_assegurar_com_defaults = {
            'prompt_version_key': default_prompt_key_retro,
            'target_run_id': default_target_run_id_retro,
            'temperatura_usada': default_temperatura_retro,
            'modelo': default_modelo_retro
        }
        logger.info("Assegurando colunas e preenchendo NaNs com defaults...")
        for col, default_val in colunas_para_assegurar_com_defaults.items():
            if col not in df.columns:
                df[col] = default_val
                logger.info(f"Coluna '{col}' adicionada com default '{default_val}'.")
            else:
                if col == 'temperatura_usada':
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(default_val)
                else:
                    df[col].fillna(default_val, inplace=True)
        for col_meta in COLUNAS_METADADOS_API:
            if col_meta not in df.columns and col_meta not in colunas_para_assegurar_com_defaults:
                 df[col_meta] = 0 if "token" in col_meta else None
        hashes_recalculados = 0
        logger.info("Recalculando hashes para todos os registros...")
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Recalculando Hashes"):
            desc = str(row.get('descrição','')).strip()
            modelo_row = str(row.get('modelo')).strip()
            temp_usada_row = row.get('temperatura_usada')
            prompt_key_row = str(row.get('prompt_version_key')).strip()
            target_run_row = str(row.get('target_run_id')).strip()
            if desc and modelo_row and prompt_key_row and target_run_row and pd.notna(temp_usada_row):
                try:
                    temp_float = float(temp_usada_row)
                    novo_hash = gerar_hash_processamento(desc, modelo_row, temp_float, prompt_key_row, target_run_row)
                    df.loc[index, 'hash_processamento'] = novo_hash
                    hashes_recalculados += 1
                except ValueError as ve:
                    logger.error(f"Valor inválido (temp: {temp_usada_row}) na linha {index} ao recalcular hash: {ve}")
                    df.loc[index, 'hash_processamento'] = None
            else:
                logger.warning(f"Dados insuficientes para recalcular hash na linha {index}: desc='{desc[:10]}...', mod='{modelo_row}', pk='{prompt_key_row}', tr='{target_run_row}', temp='{temp_usada_row}'")
                df.loc[index, 'hash_processamento'] = None
        for col_std in COLUNAS_CONTROLE_COMPLETO:
            if col_std not in df.columns:
                df[col_std] = 0 if "token" in col_std else None
                logger.info(f"Coluna padrão '{col_std}' adicionada ao DataFrame antes de salvar.")
        df = df[COLUNAS_CONTROLE_COMPLETO]
        df.to_excel(caminho_arquivo_excel, index=False)
        logger.info(f"Arquivo de controle com hashes RECALCULADOS e colunas asseguradas salvo em '{caminho_arquivo_excel}'. {hashes_recalculados} hashes recalculados/atualizados.")
        return True
    except Exception as e:
        logger.exception(f"Erro CRÍTICO ao RECALCULAR hashes no arquivo de controle: {e}")
        return False

# --- CARREGAMENTO INICIAL E ATUALIZAÇÃO OPCIONAL DE HASH ---
df_controle_historico_completo, map_hash_para_resultado_existente, set_hashes_com_erro_anterior = carregar_resultados_anteriores_completos(PATH_ARQUIVO_CONTROLE_EXCEL)
logger.info(f"Total hashes únicos carregados inicialmente: {len(map_hash_para_resultado_existente)}. Hashes com erro: {len(set_hashes_com_erro_anterior)}.")

RUN_HASH_RECALCULATION_ONCE = False
if RUN_HASH_RECALCULATION_ONCE:
    logger.warning("ATENÇÃO: INICIANDO RECALCULATE DE HASHES NO ARQUIVO DE CONTROLE PRINCIPAL. FAÇA BACKUP PRIMEIRO!")
    DEFAULT_PROMPT_KEY_PARA_ANTIGOS = "v1"
    DEFAULT_TARGET_RUN_ID_PARA_ANTIGOS = "DRI_L0size1000_Nc173_Seed1042"
    DEFAULT_TEMPERATURA_PARA_ANTIGOS = 0.2
    DEFAULT_MODELO_PARA_ANTIGOS = "gemini-1.5-flash-latest"
    success_recalc = recalcular_e_atualizar_hashes_controle(
        PATH_ARQUIVO_CONTROLE_EXCEL,
        DEFAULT_PROMPT_KEY_PARA_ANTIGOS,
        DEFAULT_TARGET_RUN_ID_PARA_ANTIGOS,
        DEFAULT_TEMPERATURA_PARA_ANTIGOS,
        DEFAULT_MODELO_PARA_ANTIGOS
    )
    if success_recalc:
        logger.info("Recalculate de hashes concluído. Recarregando dados do controle...")
        df_controle_historico_completo, map_hash_para_resultado_existente, set_hashes_com_erro_anterior = carregar_resultados_anteriores_completos(PATH_ARQUIVO_CONTROLE_EXCEL)
        logger.info(f"APÓS RECALCULATE E RECARGA - Hashes únicos: {len(map_hash_para_resultado_existente)}. Hashes com erro: {len(set_hashes_com_erro_anterior)}.")
    else: logger.error("FALHA no RECALCULATE de hashes.")
    logger.warning("RECALCULATE EXECUTADO. MUDE 'RUN_HASH_RECALCULATION_ONCE' PARA False.")


def carregar_dados_lote(caminho_base_csv: str, target_run_id_filtro: str) -> list:
    dataset_lote = []
    try:
        df_full_lote = pd.read_csv(caminho_base_csv)
        df_filtrado_lote = df_full_lote[df_full_lote['run_id'] == target_run_id_filtro].copy()
        if not df_filtrado_lote.empty:
            required_cols_lote = ['text_sample', 'true_label_sample']
            if all(col in df_filtrado_lote.columns for col in required_cols_lote):
                df_filtrado_lote.dropna(subset=required_cols_lote, inplace=True)
                dataset_lote = [
                    {"descricao_produto": str(row['text_sample']).strip(), "ground_truth_categoria": str(row['true_label_sample']).strip()}
                    for _, row in df_filtrado_lote.iterrows()
                ]
                logger.info(f"Dataset para target_run_id '{target_run_id_filtro}' carregado com {len(dataset_lote)} itens.")
            else: logger.error(f"CSV para '{target_run_id_filtro}' não contém colunas {required_cols_lote}.")
        else: logger.warning(f"Nenhuma linha para '{target_run_id_filtro}' em '{caminho_base_csv}'.")
    except FileNotFoundError: logger.error(f"Arquivo CSV base não encontrado: {caminho_base_csv} para lote {target_run_id_filtro}")
    except Exception as e: logger.exception(f"Erro ao carregar dados do lote para '{target_run_id_filtro}'.")
    return dataset_lote


# --- FUNÇÕES DE API GEMINI E PARSE ---
def make_gemini_api_call(prompt_text: str, model_name: str, temperature: float,
                         descricao_original_para_log: str = "N/A") -> genai_types.GenerateContentResponse | None:
    if not API_CONFIGURADA_COM_SUCESSO:
        logger.error(f"API não config. Chamada API pulada para '{descricao_original_para_log}'.")
        return None
    try:
        model = genai.GenerativeModel(model_name)
        generation_config = genai_types.GenerationConfig(response_mime_type="application/json", temperature=temperature)
        safety_settings = [{"category": c, "threshold": "BLOCK_MEDIUM_AND_ABOVE"} for c in ["HARM_CATEGORY_HARASSMENT", "HARM_CATEGORY_HATE_SPEECH", "HARM_CATEGORY_SEXUALLY_EXPLICIT", "HARM_CATEGORY_DANGEROUS_CONTENT"]]
        logger.info(f"Chamando API Gemini para '{descricao_original_para_log}' (Modelo: {model_name}, Temp: {temperature:.2f}).")
        response = model.generate_content(prompt_text, generation_config=generation_config, safety_settings=safety_settings)
        logger.debug(f"Resposta bruta da API para '{descricao_original_para_log}': {str(response)}")
        return response
    except Exception as e:
        logger.exception(f"Erro na chamada da API Gemini para '{descricao_original_para_log}'.")
        return None

def get_gemini_llm_output_with_metadata(prompt_text: str, descricao_original_para_log: str,
                                         model_name_para_api: str, temperatura_para_api: float,
                                         prompt_key_para_api: str, target_run_id_para_api: str
                                         ) -> dict:
    api_response_object = make_gemini_api_call(prompt_text, model_name_para_api, temperatura_para_api, descricao_original_para_log)
    error_output_dict = {
        "text_response": json.dumps({"descrição": descricao_original_para_log, "descrição_expandida": STRING_ERRO_API_NA_EXPANDIDA, "descricao_machine_learning": unidecode(STRING_ERRO_API_NA_EXPANDIDA.lower()), "categoria": "_RARE_", "racional": "Falha API ou resposta inválida."}, ensure_ascii=False),
        "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
        "modelo": model_name_para_api, 
        "temperatura_usada": temperatura_para_api,
        "prompt_version_key": prompt_key_para_api,
        "target_run_id": target_run_id_para_api
    }
    if api_response_object is None: return error_output_dict
    try:
        text_content = api_response_object.text
        prompt_tokens = api_response_object.usage_metadata.prompt_token_count
        completion_tokens = api_response_object.usage_metadata.candidates_token_count
        total_tokens = api_response_object.usage_metadata.total_token_count
        logger.info(f"API Sucesso para '{descricao_original_para_log}': Modelo: {model_name_para_api}, Temp: {temperatura_para_api:.2f}, PTokens: {prompt_tokens}, CTokens: {completion_tokens}")
        return {"text_response": text_content, "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": total_tokens, 
                "modelo": model_name_para_api, "temperatura_usada": temperatura_para_api,
                "prompt_version_key": prompt_key_para_api, "target_run_id": target_run_id_para_api}
    except Exception as e:
        logger.exception(f"Erro ao processar resposta da API para '{descricao_original_para_log}'.")
        error_output_dict["text_response"] = json.dumps({"descrição": descricao_original_para_log, "descrição_expandida": STRING_ERRO_API_NA_EXPANDIDA, "descricao_machine_learning": unidecode(STRING_ERRO_API_NA_EXPANDIDA.lower()), "categoria": "_RARE_", "racional": f"Erro proc. resp API: {str(e)[:100]}"}, ensure_ascii=False)
        return error_output_dict

def parse_e_validar_resposta_especifica(response_text_str: str, descricao_original: str,
                                         categorias_mestra_para_validacao: list) -> dict:
    parsed_llm_fields = {}
    default_on_error_parse = {"descrição": descricao_original, "descrição_expandida": STRING_ERRO_API_NA_EXPANDIDA, "descricao_machine_learning": unidecode(STRING_ERRO_API_NA_EXPANDIDA.lower()), "categoria": "_RARE_", "racional": "Falha decode JSON LLM."}
    try:
        parsed_llm_fields = json.loads(response_text_str)
        if not isinstance(parsed_llm_fields, dict):
            logger.warning(f"Resp LLM não é dict JSON para '{descricao_original}'."); parsed_llm_fields = deepcopy(default_on_error_parse); parsed_llm_fields["racional"] += " Não era dict."
    except json.JSONDecodeError as e:
        logger.error(f"Erro decode JSON LLM para '{descricao_original}': {e}. Resp: {str(response_text_str)[:200]}")
        match = re.search(r"```json\s*([\s\S]*?)\s*```", response_text_str)
        if match:
            try: parsed_llm_fields = json.loads(match.group(1));
            except: parsed_llm_fields = None
        if parsed_llm_fields is None or not isinstance(parsed_llm_fields, dict): parsed_llm_fields = deepcopy(default_on_error_parse); parsed_llm_fields["racional"] += f" JSONDecodeError: {e}."
    
    parsed_llm_fields["descrição"] = descricao_original
    
    for key in COLUNAS_RESULTADO_LLM_BASE:
        if key not in parsed_llm_fields or pd.isna(parsed_llm_fields.get(key)):
            if key == "categoria": parsed_llm_fields[key] = "_RARE_"
            elif key == "descrição_expandida": parsed_llm_fields[key] = descricao_original + " (Exp. padrão por ausência)"
            elif key == "descricao_machine_learning": parsed_llm_fields[key] = unidecode(parsed_llm_fields.get("descrição_expandida", "").lower())
            elif key == "racional": parsed_llm_fields[key] = "Racional padrão (chave LLM ausente ou NaN)."
            elif key != "descrição": parsed_llm_fields[key] = "VALOR AUSENTE LLM"
            if key != "racional" and "racional" in parsed_llm_fields:
                 parsed_llm_fields["racional"] = (str(parsed_llm_fields.get("racional","")) + f" (ALERTA: Chave LLM '{key}' ausente/NaN).").strip()

    if parsed_llm_fields.get("categoria") not in categorias_mestra_para_validacao:
        original_cat_llm = parsed_llm_fields.get("categoria"); 
        logger.warning(f"Categoria '{original_cat_llm}' do LLM para '{descricao_original}' inválida (não na lista mestra). Forçando '_RARE_'.")
        parsed_llm_fields["categoria"] = "_RARE_"
        parsed_llm_fields["racional"] = (str(parsed_llm_fields.get("racional", "")) + f" Cat.LLM '{original_cat_llm}' inválida (não na mestra)->'_RARE_'.").strip()
    return parsed_llm_fields

# --- FUNÇÃO DE PROCESSAMENTO DO ITEM ---
def processar_item_descricao_produto(descricao_produto_item: str,
                                     prompt_template_para_api: str,
                                     lista_categorias_str_mestra_param: str,
                                     modelo_experimento_atual: str,
                                     temperatura_experimento_atual: float,
                                     prompt_key_experimento_atual: str,
                                     target_run_id_experimento_atual: str,
                                     categorias_mestra_py_para_validacao: list
                                     ) -> dict:
    prompt_formatado_api = prompt_template_para_api.format(
        lista_categorias_str=lista_categorias_str_mestra_param,
        descricao_produto=descricao_produto_item.strip()
    )
    gemini_output_with_meta = get_gemini_llm_output_with_metadata(
        prompt_formatado_api, 
        descricao_produto_item,
        modelo_experimento_atual,
        temperatura_experimento_atual,
        prompt_key_experimento_atual,
        target_run_id_experimento_atual
    )
    parsed_content_especifico = parse_e_validar_resposta_especifica(
        gemini_output_with_meta["text_response"],
        descricao_produto_item,
        categorias_mestra_py_para_validacao
    )
    resultado_final_combinado = deepcopy(parsed_content_especifico)
    for meta_key in ['prompt_tokens', 'completion_tokens', 'total_tokens', 'modelo', 'temperatura_usada', 'prompt_version_key', 'target_run_id']:
        resultado_final_combinado[meta_key] = gemini_output_with_meta[meta_key]
    
    hash_do_processamento = gerar_hash_processamento(
        descricao_produto_item,
        gemini_output_with_meta["modelo"],
        gemini_output_with_meta["temperatura_usada"],
        gemini_output_with_meta["prompt_version_key"],
        gemini_output_with_meta["target_run_id"]
    )
    resultado_final_combinado["hash_processamento"] = hash_do_processamento
    return resultado_final_combinado

# --- LOOP DE PROCESSAMENTO PRINCIPAL POR EXPERIMENTOS ---
todos_os_novos_resultados_nesta_execucao = []
lista_sumarios_metricas = []
lista_erros_modelo_todos_experimentos = []

experimentos_a_rodar = carregar_experimentos(PATH_ARQUIVO_EXPERIMENTOS_JSON)

if not experimentos_a_rodar:
    logger.error("Nenhum experimento definido ou erro ao carregar. Encerrando.")
else:
    logger.info(f"Iniciando {len(experimentos_a_rodar)} experimentos.")
    for id_experimento, experimento_config in enumerate(experimentos_a_rodar):
        MODELO_EXP_ATUAL = experimento_config['model_name']
        TEMP_EXP_ATUAL = experimento_config['temperature']
        PROMPT_KEY_EXP_ATUAL = experimento_config['prompt_version_key']
        TARGET_RUN_ID_EXP_ATUAL = experimento_config['target_run_id']

        logger.info(f"--- Iniciando Exp {id_experimento + 1}/{len(experimentos_a_rodar)}: Mod={MODELO_EXP_ATUAL}, Temp={TEMP_EXP_ATUAL:.2f}, PromptKey='{PROMPT_KEY_EXP_ATUAL}', Lote='{TARGET_RUN_ID_EXP_ATUAL}' ---")

        PROMPT_TEMPLATE_EXP_ATUAL = PROMPTS_ORACULO.get(PROMPT_KEY_EXP_ATUAL)
        if not PROMPT_TEMPLATE_EXP_ATUAL:
            logger.error(f"Prompt com chave '{PROMPT_KEY_EXP_ATUAL}' não encontrado. Pulando experimento.")
            lista_sumarios_metricas.append({'modelo': MODELO_EXP_ATUAL, 'temperatura': TEMP_EXP_ATUAL, 'prompt_version_key': PROMPT_KEY_EXP_ATUAL, 'target_run_id': TARGET_RUN_ID_EXP_ATUAL, 'acuracia': 0.0, 'f1_score_macro': 0.0, 'total_itens_avaliados': 0, 'itens_processados_api': 0, 'itens_reutilizados_cache': 0, 'observacao': f"Erro: Prompt Key '{PROMPT_KEY_EXP_ATUAL}' não encontrado."})
            continue

        dataset_avaliacao_lote_atual = carregar_dados_lote(PATH_BASE_CSV_DADOS_COLDSTART, TARGET_RUN_ID_EXP_ATUAL)
        if not dataset_avaliacao_lote_atual:
            logger.error(f"Dataset para lote '{TARGET_RUN_ID_EXP_ATUAL}' vazio. Pulando experimento.")
            lista_sumarios_metricas.append({'modelo': MODELO_EXP_ATUAL, 'temperatura': TEMP_EXP_ATUAL, 'prompt_version_key': PROMPT_KEY_EXP_ATUAL, 'target_run_id': TARGET_RUN_ID_EXP_ATUAL, 'acuracia': 0.0, 'f1_score_macro': 0.0, 'total_itens_avaliados': 0, 'itens_processados_api': 0, 'itens_reutilizados_cache': 0, 'observacao': f"Erro: Dataset para lote '{TARGET_RUN_ID_EXP_ATUAL}' não encontrado/vazio."})
            continue
            
        resultados_llm_exp = []
        predicoes_exp = []
        verdadeiros_exp = []
        itens_api_exp = 0
        itens_cache_exp = 0
        
        if not API_CONFIGURADA_COM_SUCESSO and not map_hash_para_resultado_existente:
             logger.error(f"API não config. e nenhum dado de controle. Pulando exp: {MODELO_EXP_ATUAL}|T{TEMP_EXP_ATUAL:.1f}|P'{PROMPT_KEY_EXP_ATUAL}'|L'{TARGET_RUN_ID_EXP_ATUAL}'")
             continue

        desc_tqdm = f"Exp {id_experimento+1} ({MODELO_EXP_ATUAL[:10]}|T{TEMP_EXP_ATUAL:.1f}|P'{PROMPT_KEY_EXP_ATUAL}'|L'{TARGET_RUN_ID_EXP_ATUAL[:10]}')"
        with tqdm(total=len(dataset_avaliacao_lote_atual), desc=desc_tqdm) as pbar:
            for item_aval in dataset_avaliacao_lote_atual:
                desc_prod = item_aval["descricao_produto"]
                gt_cat = item_aval["ground_truth_categoria"]
                
                hash_item = gerar_hash_processamento(desc_prod, MODELO_EXP_ATUAL, TEMP_EXP_ATUAL, PROMPT_KEY_EXP_ATUAL, TARGET_RUN_ID_EXP_ATUAL)
                resultado_item_final = None; reutilizar = False

                if hash_item in map_hash_para_resultado_existente:
                    if hash_item not in set_hashes_com_erro_anterior:
                        resultado_item_final = deepcopy(map_hash_para_resultado_existente[hash_item])
                        resultado_item_final.pop('ground_truth_categoria_controle', None); resultado_item_final.pop('predicao_categoria_controle', None)
                        reutilizar = True; itens_cache_exp += 1
                    else: logger.info(f"Hash {hash_item[:7]} de '{desc_prod[:20]}...' era erro. Reprocessando.")
                
                if not reutilizar:
                    if not API_CONFIGURADA_COM_SUCESSO:
                        logger.warning(f"API não config. para '{desc_prod[:20]}'. Usando erro padrão.")
                        resultado_item_final = {"descrição": desc_prod, "descrição_expandida": STRING_ERRO_API_NA_EXPANDIDA, "descricao_machine_learning": unidecode(STRING_ERRO_API_NA_EXPANDIDA.lower()), "categoria": "_RARE_", "racional": "API não configurada.", "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "modelo": MODELO_EXP_ATUAL, "temperatura_usada": TEMP_EXP_ATUAL, "prompt_version_key": PROMPT_KEY_EXP_ATUAL, "target_run_id": TARGET_RUN_ID_EXP_ATUAL, "hash_processamento": hash_item}
                    else:
                        resultado_item_final = processar_item_descricao_produto(
                            desc_prod, PROMPT_TEMPLATE_EXP_ATUAL, 
                            lista_categorias_str_mestra,
                            MODELO_EXP_ATUAL, TEMP_EXP_ATUAL, 
                            PROMPT_KEY_EXP_ATUAL, TARGET_RUN_ID_EXP_ATUAL,
                            lista_categorias_py_mestra
                        )
                        itens_api_exp += 1
                        todos_os_novos_resultados_nesta_execucao.append(deepcopy(resultado_item_final))
                        if itens_api_exp > 0 and itens_api_exp % 30 == 0: 
                            logger.info(f"Pausa de 1s após {itens_api_exp} chamadas API no experimento.")
                            time.sleep(1)
                
                resultados_llm_exp.append(resultado_item_final)
                pred_cat = resultado_item_final.get("categoria", "_RARE_")
                predicoes_exp.append(pred_cat)
                verdadeiros_exp.append(gt_cat)

                if pred_cat != gt_cat:
                    lista_erros_modelo_todos_experimentos.append({
                        'descrição': desc_prod, 'ground_truth_categoria': gt_cat, 'predicao_categoria': pred_cat,
                        'racional': resultado_item_final.get('racional', ''), 'modelo': MODELO_EXP_ATUAL,
                        'temperatura_usada': TEMP_EXP_ATUAL, 'prompt_version_key': PROMPT_KEY_EXP_ATUAL,
                        'target_run_id': TARGET_RUN_ID_EXP_ATUAL, 'hash_processamento': hash_item
                    })
                pbar.update(1)
        
        logger.info(f"Exp {desc_tqdm} concluído. API: {itens_api_exp}, Cache: {itens_cache_exp}")

        if verdadeiros_exp and predicoes_exp:
            labels_para_f1 = sorted(list(set(lista_categorias_py_mestra + verdadeiros_exp + predicoes_exp)))
            acc = accuracy_score(verdadeiros_exp, predicoes_exp)
            f1_macro = f1_score(verdadeiros_exp, predicoes_exp, average='macro', zero_division=0, labels=labels_para_f1)
            lista_sumarios_metricas.append({'modelo': MODELO_EXP_ATUAL, 'temperatura': TEMP_EXP_ATUAL, 'prompt_version_key': PROMPT_KEY_EXP_ATUAL, 'target_run_id': TARGET_RUN_ID_EXP_ATUAL, 'acuracia': acc, 'f1_score_macro': f1_macro, 'total_itens_avaliados': len(verdadeiros_exp), 'itens_processados_api': itens_api_exp, 'itens_reutilizados_cache': itens_cache_exp})
            logger.info(f"Métricas para Exp {desc_tqdm}: Acurácia={acc:.4f}, F1-Macro={f1_macro:.4f}")
        else:
            logger.warning(f"Não foi possível calcular métricas para o exp {desc_tqdm}.")
            lista_sumarios_metricas.append({'modelo': MODELO_EXP_ATUAL, 'temperatura': TEMP_EXP_ATUAL, 'prompt_version_key': PROMPT_KEY_EXP_ATUAL, 'target_run_id': TARGET_RUN_ID_EXP_ATUAL, 'acuracia': 0.0, 'f1_score_macro': 0.0, 'total_itens_avaliados': 0, 'itens_processados_api': itens_api_exp, 'itens_reutilizados_cache': itens_cache_exp, 'observacao': 'Erro ou nenhum item avaliado'})
    logger.info("Todos os experimentos foram concluídos.")


# --- SALVAMENTO DOS RESULTADOS AGREGADOS E ARQUIVOS DE ANÁLISE ---
if todos_os_novos_resultados_nesta_execucao or not df_controle_historico_completo.empty :
    df_para_salvar_controle = df_controle_historico_completo.copy()
    if todos_os_novos_resultados_nesta_execucao:
        df_novos_resultados = pd.DataFrame(todos_os_novos_resultados_nesta_execucao)
        for col_control in COLUNAS_CONTROLE_COMPLETO:
            if col_control not in df_novos_resultados.columns:
                df_novos_resultados[col_control] = None
        df_para_salvar_controle = pd.concat([df_para_salvar_controle, df_novos_resultados[COLUNAS_CONTROLE_COMPLETO]], ignore_index=True)
    
    for col_token in ['prompt_tokens', 'completion_tokens', 'total_tokens']:
        df_para_salvar_controle[col_token] = pd.to_numeric(df_para_salvar_controle[col_token], errors='coerce').fillna(0).astype(int)
    df_para_salvar_controle['temperatura_usada'] = pd.to_numeric(df_para_salvar_controle['temperatura_usada'], errors='coerce').fillna(0.0).astype(float)

    logger.info(f"Antes de drop_duplicates no controle final: {len(df_para_salvar_controle)} linhas.")
    df_para_salvar_controle.drop_duplicates(subset=['hash_processamento'], keep='last', inplace=True)
    logger.info(f"Depois de drop_duplicates no controle final: {len(df_para_salvar_controle)} linhas.")
    
    for col_final in COLUNAS_CONTROLE_COMPLETO: # Assegurar todas as colunas padrão
        if col_final not in df_para_salvar_controle.columns:
            df_para_salvar_controle[col_final] = 0 if "token" in col_final else (0.0 if "temperatura" in col_final else None)
    df_para_salvar_controle = df_para_salvar_controle[COLUNAS_CONTROLE_COMPLETO] # Reordenar

    try:
        # Criar diretório se não existir
        os.makedirs(os.path.dirname(PATH_ARQUIVO_CONTROLE_EXCEL), exist_ok=True)
        df_para_salvar_controle.to_excel(PATH_ARQUIVO_CONTROLE_EXCEL, index=False)
        logger.info(f"Arquivo de controle '{PATH_ARQUIVO_CONTROLE_EXCEL}' salvo/atualizado com {len(df_para_salvar_controle)} registros.")
    except Exception as e:
        logger.exception(f"Erro ao salvar controle principal '{PATH_ARQUIVO_CONTROLE_EXCEL}'.")
else:
    logger.info("Nenhum resultado novo e nenhum histórico para salvar no controle principal.")

if lista_sumarios_metricas:
    df_sumario_metricas = pd.DataFrame(lista_sumarios_metricas)
    try:
        os.makedirs(os.path.dirname(PATH_ARQUIVO_SUMARIO_METRICAS_EXCEL), exist_ok=True)
        df_sumario_metricas.to_excel(PATH_ARQUIVO_SUMARIO_METRICAS_EXCEL, index=False)
        logger.info(f"Sumário de métricas salvo em '{PATH_ARQUIVO_SUMARIO_METRICAS_EXCEL}'.")
        if 'display' in globals() and callable(display): display(df_sumario_metricas) 
    except Exception as e: logger.exception(f"Erro ao salvar sumário de métricas '{PATH_ARQUIVO_SUMARIO_METRICAS_EXCEL}'.")
else: logger.warning("Nenhum sumário de métricas para salvar.")

if lista_erros_modelo_todos_experimentos:
    df_erros_modelo = pd.DataFrame(lista_erros_modelo_todos_experimentos)
    df_erros_modelo.sort_values(by=['modelo', 'temperatura_usada', 'prompt_version_key', 'target_run_id', 'descrição'], inplace=True)
    try:
        os.makedirs(os.path.dirname(PATH_ARQUIVO_ERROS_MODELO_EXCEL), exist_ok=True)
        df_erros_modelo.to_excel(PATH_ARQUIVO_ERROS_MODELO_EXCEL, index=False)
        logger.info(f"Arquivo de erros do modelo salvo em '{PATH_ARQUIVO_ERROS_MODELO_EXCEL}' com {len(df_erros_modelo)} erros.")
    except Exception as e: logger.exception(f"Erro ao salvar arquivo de erros '{PATH_ARQUIVO_ERROS_MODELO_EXCEL}'.")
else: logger.info("Nenhum erro de modelo registrado nesta execução.")
logger.info("--- FIM DA EXECUÇÃO ---")